## Library import


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
import warnings
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix,classification_report
warnings.filterwarnings('ignore')

## Data import


In [2]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/06_CHURN_PREDICTION/Churn_Modelling.csv')

## EDA(exploratory data analysis)


In [3]:
df.columns

Index(['RowNumber', 'CustomerId', 'Surname', 'CreditScore', 'Geography',
       'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited'],
      dtype='object')

In [4]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


RowNumber,customerID,Surname columns is not useful so drop it

In [6]:
df.drop(['RowNumber','CustomerId','Surname'],axis = 1,inplace = True)

In [7]:
df.shape

(10000, 11)

In [8]:
df.columns

Index(['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary',
       'Exited'],
      dtype='object')

In [9]:
df.Exited.value_counts()

,count
Exited,
0,7963
1,2037


In [10]:
df.dtypes

,0
CreditScore,int64
Geography,object
Gender,object
Age,int64
Tenure,int64
Balance,float64
NumOfProducts,int64
HasCrCard,int64
IsActiveMember,int64
EstimatedSalary,float64


In [11]:
def print_unique_val(df):
    for col in df:
        if df[col].dtypes == 'object':
            print(f'{col}: {df[col].unique()}')

In [12]:
print_unique_val(df)

Geography: ['France' 'Spain' 'Germany']
Gender: ['Female' 'Male']


In [13]:
df1 = pd.get_dummies(data = df,
                     columns = ['Geography',
                                'Gender'
                                ]).astype('int')
df1.columns

Index(['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard',
       'IsActiveMember', 'EstimatedSalary', 'Exited', 'Geography_France',
       'Geography_Germany', 'Geography_Spain', 'Gender_Female', 'Gender_Male'],
      dtype='object')

In [14]:
df1.sample(5)

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain,Gender_Female,Gender_Male
7427,714,33,8,122017,1,0,0,162515,0,0,0,1,0,1
3126,656,43,7,134919,1,1,0,194691,0,1,0,0,0,1
1785,713,40,3,114446,2,1,1,87308,0,0,1,0,0,1
5482,603,46,2,0,2,1,0,174478,0,0,0,1,0,1
5543,710,38,3,130588,1,1,1,154997,0,1,0,0,0,1


In [15]:
df1.dtypes

,0
CreditScore,int64
Age,int64
Tenure,int64
Balance,int64
NumOfProducts,int64
HasCrCard,int64
IsActiveMember,int64
EstimatedSalary,int64
Exited,int64
Geography_France,int64


In [17]:
cols_to_scale = ['CreditScore','Balance','EstimatedSalary']

In [18]:
scaler = MinMaxScaler()
df1[cols_to_scale] = scaler.fit_transform(df1[cols_to_scale])

In [19]:
x = df1.drop('Exited',axis =1)
y = df1.Exited

In [21]:
df1.sample(1)

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain,Gender_Female,Gender_Male
5906,0.876,32,4,0.446711,1,0,0,0.446827,0,1,0,0,0,1


## Model Building

In [22]:
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.2,
                                random_state = 99)

In [23]:
x_train.shape

(8000, 13)

In [24]:
x_test.shape

(2000, 13)

In [25]:
y_train.value_counts()

,count
Exited,
0,6386
1,1614


In [26]:
y_test.value_counts()

,count
Exited,
0,1577
1,423


In [27]:
def ANN(x_train,y_train,x_test,y_test,loss,weights):
    model = keras.Sequential([
        keras.layers.Dense(13,input_dim = 13,activation = 'relu'),
        keras.layers.Dense(10,activation = 'relu'),
        keras.layers.Dense(1,activation = 'sigmoid')
    ])
    model.compile(optimizer = 'adam',loss=loss,metrics=['accuracy'])

    if weights == -1:
        model.fit(x_train,y_train,epochs = 100)
    else:
        model.fit(x_train,y_train,epochs = 100,class_weight = weights)

    print('Loss and Accuracy: \n',model.evaluate(x_test,y_test))

    y_preds = model.predict(x_test)
    y_preds = np.round(y_preds)

    print('classification Report: \n',classification_report(y_test,y_preds))
    return y_preds


In [28]:
y_preds = ANN(x_train,y_train,x_test,y_test,'binary_crossentropy',-1)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.4404 - loss: 6.4498
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7952 - loss: 0.5012
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8037 - loss: 0.4687
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8066 - loss: 0.4540
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8087 - loss: 0.4439
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8118 - loss: 0.4370
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8075 - loss: 0.4426
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8112 - loss: 0.4368
Epoch 9/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7994 - loss: 0.4544
Epoch 10/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8046 - loss: 0.4386
Epoch 11/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8138 - loss: 0.4320
Epoch 12/100
250/250 ━━━━━━━━━━━━━━━━━━━━

## Handling Imbalance


### Under Sampling

In [29]:
df1.Exited.value_counts()

,count
Exited,
0,7963
1,2037


In [30]:
count_class_0,count_class_1 = df1.Exited.value_counts()
df_class_0 = df1[df1.Exited == 0]
df_class_1 = df1[df1.Exited == 1]

In [31]:
df_class_0.shape

(7963, 14)

In [32]:
df_class_1.shape

(2037, 14)

In [33]:
df_class_0.sample(2)

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain,Gender_Female,Gender_Male
1476,0.812,26,5,0.405109,2,0,1,0.772318,0,1,0,0,1,0
9065,0.412,30,7,0.000000,2,1,1,0.933274,0,1,0,0,1,0


In [34]:
df_class_0_undersample = df_class_0.sample(count_class_1)

In [35]:
df_test_undersampling = pd.concat([df_class_0_undersample,df_class_1],axis = 0)

In [37]:
print('UNDER SAMPLED SHAPE:')
df_test_undersampling.Exited.value_counts()

UNDER SAMPLED SHAPE:


,count
Exited,
0,2037
1,2037


In [38]:
x_u = df_test_undersampling.drop('Exited',axis ='columns')
y_u = df_test_undersampling.Exited

In [39]:
x_u.shape

(4074, 13)

In [40]:
y_u.shape

(4074,)

In [41]:
x_train_u,x_test_u,y_train_u,y_test_u = train_test_split(x_u,y_u,test_size = 0.2, random_state =15, stratify = y_u)

In [42]:
y_train_u.value_counts()

,count
Exited,
1,1630
0,1629


In [43]:
y_test_u.value_counts()

,count
Exited,
0,408
1,407


In [44]:
y_preds = ANN(x_train_u,y_train_u,x_test_u,y_test_u,'binary_crossentropy',-1)

Epoch 1/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5266 - loss: 2.5451
Epoch 2/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5877 - loss: 0.6818
Epoch 3/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6465 - loss: 0.6401
Epoch 4/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6763 - loss: 0.6121
Epoch 5/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6516 - loss: 0.6230
Epoch 6/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6794 - loss: 0.6096
Epoch 7/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6893 - loss: 0.5908
Epoch 8/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6876 - loss: 0.5956
Epoch 9/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6860 - loss: 0.5869
Epoch 10/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6685 - loss: 0.6055
Epoch 11/100
102/102 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6885 - loss: 0.5870
Epoch 12/100
102/102 ━━━━━━━━━━━━━━━━━━━━

### Over Sampling

In [45]:
count_class_0

7963

In [46]:
count_class_1

2037

In [47]:
df_class_1_over = df_class_1.sample(count_class_0,replace=True)

In [48]:
df_class_1_over.shape

(7963, 14)

In [49]:
df_test_oversample = pd.concat([df_class_0,df_class_1_over],axis = 0)

In [50]:
df_test_oversample.shape

(15926, 14)

In [52]:
print('After Over Sampling:')
df_test_oversample.Exited.value_counts()

After Over Sampling:


,count
Exited,
0,7963
1,7963


In [53]:
x_o = df_test_oversample.drop('Exited',axis = 'columns')
y_o = df_test_oversample.Exited

In [55]:
x_o.shape

(15926, 13)

In [56]:
y_o.shape

(15926,)

In [57]:
x_train_o,x_test_o,y_train_o,y_test_o = train_test_split(x_o,y_o,test_size= 0.2, random_state=15,stratify = y_o)

In [58]:
y_train_o.value_counts()

,count
Exited,
0,6370
1,6370


In [59]:
y_test_o.value_counts()

,count
Exited,
0,1593
1,1593


In [60]:
y_preds = ANN(x_train_o,y_train_o,x_test_o,y_test_o,'binary_crossentropy',-1)

Epoch 1/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.5345 - loss: 0.6840
Epoch 2/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6651 - loss: 0.6264
Epoch 3/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7010 - loss: 0.5869
Epoch 4/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6961 - loss: 0.5872
Epoch 5/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7000 - loss: 0.5787
Epoch 6/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7065 - loss: 0.5727
Epoch 7/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7095 - loss: 0.5684
Epoch 8/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7086 - loss: 0.5669
Epoch 9/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.7276 - loss: 0.5514
Epoch 10/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.7337 - loss: 0.5412
Epoch 11/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.7388 - loss: 0.5411
Epoch 12/100
399/399 ━━━━━━━━━━━━━━━━━━━━

### SMOTE

In [61]:
from imblearn.over_sampling import SMOTE

In [62]:
df1.sample(2)

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain,Gender_Female,Gender_Male
7650,0.80,42,8,0.605170,2,1,0,0.343453,0,0,1,0,0,1
3474,0.74,33,6,0.387361,1,0,0,0.459394,0,0,0,1,0,1


In [63]:
df1.shape

(10000, 14)

In [65]:
x = df1.drop('Exited',axis = 1)
y = df1.Exited

In [66]:
y.value_counts()

,count
Exited,
0,7963
1,2037


In [67]:
smote = SMOTE(sampling_strategy='minority')
x_sm,y_sm = smote.fit_resample(x,y)

In [68]:
y_sm.value_counts()

,count
Exited,
1,7963
0,7963


In [69]:
x_train_sm,x_test_sm,y_train_sm,y_test_sm = train_test_split(x_sm,y_sm,
                    test_size = 0.2, random_state = 15, stratify = y_sm)

In [70]:
y_train_sm.value_counts()

,count
Exited,
0,6370
1,6370


In [71]:
y_test_sm.value_counts()

,count
Exited,
0,1593
1,1593


In [72]:
y_preds = ANN(x_train_sm,y_train_sm,x_test_sm,y_test_sm,'binary_crossentropy',-1)

Epoch 1/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.5375 - loss: 1.3805
Epoch 2/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7350 - loss: 0.5322
Epoch 3/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7553 - loss: 0.5024
Epoch 4/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7599 - loss: 0.5011
Epoch 5/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7732 - loss: 0.4865
Epoch 6/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7693 - loss: 0.4848
Epoch 7/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7631 - loss: 0.4968
Epoch 8/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.7655 - loss: 0.4867
Epoch 9/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7765 - loss: 0.4755
Epoch 10/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7723 - loss: 0.4802
Epoch 11/100
399/399 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7760 - loss: 0.4765
Epoch 12/100
399/399 ━━━━━━━━━━━━━━━━━━━━

### Ensemble with Undersampling

In [73]:
df.Exited.value_counts()

,count
Exited,
0,7963
1,2037


In [74]:
x = df1.drop('Exited',axis = 1)
y = df1.Exited

In [75]:
x_train_e,x_test_e,y_train_e,y_test_e = train_test_split(x,y,test_size=0.2,random_state = 15,stratify = y)

In [76]:
y_train_e.value_counts()

,count
Exited,
0,6370
1,1630


model1 --> class1(1495) + class0(0, 1495)

model2 --> class1(1495) + class0(1496, 2990)

model3 --> class1(1495) + class0(2990, 4130)

In [77]:
df3 = x_train_e.copy()
df3['Churn'] = y_train_e

In [78]:
df3.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain,Gender_Female,Gender_Male,Churn
5710,0.856,34,5,0.554265,2,0,0,0.339722,1,0,0,0,1,0
3745,0.852,37,1,0.371163,2,1,1,0.980433,0,1,0,1,0,0
5429,0.664,48,7,0.000000,2,1,0,0.325321,1,0,0,1,0,0
551,0.648,47,6,0.426074,1,1,1,0.010341,0,1,0,0,1,1
8967,0.970,25,7,0.000000,2,1,1,0.417230,1,0,0,0,1,0


In [80]:
df3_class_0 = df1[df1.Exited == 0]
df3_class_1 = df1[df1.Exited == 1]

In [81]:
df3_class_0.Exited.value_counts()

,count
Exited,
0,7963


In [82]:
df3_class_1.Exited.value_counts()

,count
Exited,
1,2037


In [85]:
def get_train_batch(df_major,df_minor,start,end):
    df_train = pd.concat([df_major[start:end],df_minor],axis = 0)
    x_train = df_train.drop('Exited',axis = 'columns')
    y_train = df_train.Exited
    return x_train,y_train

In [86]:
x_train_ens,y_train_ens = get_train_batch(df3_class_0,df3_class_1,0,1495)

y_preds1 = ANN(x_train_ens,y_train_ens,x_test_e,y_test_e,'binary_crossentropy',-1)

Epoch 1/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.5162 - loss: 0.9238
Epoch 2/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5797 - loss: 0.6605
Epoch 3/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6176 - loss: 0.6372
Epoch 4/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6712 - loss: 0.6179
Epoch 5/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6768 - loss: 0.6028
Epoch 6/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6919 - loss: 0.5903
Epoch 7/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6947 - loss: 0.5802
Epoch 8/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6990 - loss: 0.5803
Epoch 9/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6983 - loss: 0.5811
Epoch 10/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7050 - loss: 0.5701
Epoch 11/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7133 - loss: 0.5672
Epoch 12/100
111/111 ━━━━━━━━━━━━━━━━━━━━

In [87]:
x_train_ens,y_train_ens = get_train_batch(df3_class_0,df3_class_1,1495,2990)

y_preds2 = ANN(x_train_ens,y_train_ens,x_test_e,y_test_e,'binary_crossentropy',-1)

Epoch 1/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.5443 - loss: 1.0812
Epoch 2/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5706 - loss: 0.6670
Epoch 3/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6034 - loss: 0.6515
Epoch 4/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6252 - loss: 0.6423
Epoch 5/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6548 - loss: 0.6263
Epoch 6/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6901 - loss: 0.6074
Epoch 7/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6681 - loss: 0.6036
Epoch 8/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6791 - loss: 0.5929
Epoch 9/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6961 - loss: 0.5846
Epoch 10/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6883 - loss: 0.5920
Epoch 11/100
111/111 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6921 - loss: 0.5814
Epoch 12/100
111/111 ━━━━━━━━━━━━━━━━━━━━

In [88]:
x_train_ens,y_train_ens = get_train_batch(df3_class_0,df3_class_1,2990,4130)

y_preds3 = ANN(x_train_ens,y_train_ens,x_test_e,y_test_e,'binary_crossentropy',-1)

Epoch 1/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6343 - loss: 1.6345
Epoch 2/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6691 - loss: 0.6099
Epoch 3/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6752 - loss: 0.6120
Epoch 4/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6672 - loss: 0.6032
Epoch 5/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6768 - loss: 0.6016
Epoch 6/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6840 - loss: 0.5928
Epoch 7/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6831 - loss: 0.5900
Epoch 8/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7013 - loss: 0.5754
Epoch 9/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7079 - loss: 0.5619
Epoch 10/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7057 - loss: 0.5710
Epoch 11/100
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6830 - loss: 0.5772
Epoch 12/100
100/100 ━━━━━━━━━━━━━━━━━━━━

In [89]:
len(y_preds1),len(y_preds2),len(y_preds3)

(2000, 2000, 2000)

In [ ]:
# vote 1 + vote 2 + vote 3
# 0 + 0 + 1 = 1 majority is 0
# 1 + 1 + 0 = 2 majority is 1
# 0 + 0 + 0 = 0 majority is 0
# 1 + 1 + 1 = 1 majority is 1
# so that sum of 3 predictions less than or equal to 1 the majority is 0
# if greater than 1 the majority is said to be 1

In [90]:
y_pred_final = y_preds1.copy()

for i in range(len(y_preds1)):
    #print(y_preds1[i],y_preds2[i],y_preds3[i])
    n_ones = y_preds1[i]+y_preds2[i]+y_preds3[i]
    if n_ones > 1:
        y_pred_final[i] = 1
    else:
        y_pred_final[i] = 0

In [91]:
print(classification_report(y_test_e,y_pred_final))

              precision    recall  f1-score   support

           0       0.91      0.64      0.75      1593
           1       0.35      0.76      0.48       407

    accuracy                           0.66      2000
   macro avg       0.63      0.70      0.61      2000
weighted avg       0.80      0.66      0.69      2000

